# Day 1 Lab — Build the Research Agent Yourself

Fill in every TODO. Each step tells you exactly **where in the LangGraph docs to look**. Don't open the solution until you've tried each step — the point of Day 1 is learning to **think in state graphs**.

```
START → collect → store_memory → analyze → evaluate
           ↑                                  │
           └── quality < 7 (max 3 tries) ─────┤
                                              └ quality >= 7
                                                    ↓
                                       report → audit → END
```

**Read before you start (~30 min):**
1. [Thinking in LangGraph](https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph) — the mental model
2. [Graph API concepts](https://docs.langchain.com/oss/python/langgraph/graph-api) — State, Nodes, Edges
3. [Using the Graph API](https://docs.langchain.com/oss/python/langgraph/use-graph-api) — code patterns you'll copy

API reference: https://reference.langchain.com/python/langgraph/

**Setup:** `uv sync`, then create `.env` (or set `USE_FAKE=1` — see README.md). Launch with `uv run jupyter lab`.

In [78]:
import os
import operator
from datetime import datetime
from typing import Annotated, List, Dict
from typing_extensions import TypedDict

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_core.messages import HumanMessage

# TODO STEP 0 — import the graph building blocks from langgraph.
# You need: StateGraph, START, END from langgraph.graph
#           InMemorySaver from langgraph.checkpoint.memory
# WHERE TO LOOK: "Graph API" docs, first code example on the page.
# from langgraph.graph import ...
# from langgraph.checkpoint.memory import ...
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()


True

## STEP 1 — THE STATE  (the "digital clipboard" from the slides)
Define a TypedDict with everything the workflow needs to remember:
  topic (str), search_query (str), collected_data (List[Dict]),
  analyzed_data (List[Dict]), quality_score (int),
  iteration_count (int), final_report (str), execution_logs

KEY IDEA: execution_logs should use a REDUCER so every node can
APPEND log lines instead of overwriting the list:
    execution_logs: Annotated[List[str], operator.add]

WHERE TO LOOK: Graph API docs → "State" section → "Reducers".
  https://docs.langchain.com/oss/python/langgraph/graph-api
ASK YOURSELF: what happens to a plain (non-reducer) key when two
nodes write it? What happens with operator.add?

In [79]:
from typing import List, Dict, TypedDict, Annotated
import operator

class AgentState(TypedDict):
    topic: str
    search_query: str
    collected_data: List[Dict]
    analyzed_data: List[Dict]
    quality_score: int
    iteration_count: int
    final_report: str
    execution_logs: Annotated[List[str], operator.add]


## STEP 2 — MODEL, SEARCH TOOL, EMBEDDINGS
Create:
  llm          = ChatOpenAI(model="gpt-4o-mini", temperature=0)
  search_tool  = TavilySearch(max_results=5)   # langchain_tavily!
  vector_store = a Chroma or InMemoryVectorStore with embeddings

------------------------------------------------------------
USING OPENROUTER (free models — recommended for this course)
------------------------------------------------------------
OpenRouter is OpenAI-compatible, so ChatOpenAI works as-is —
you only change the key, the base_url, and the model name.

1. Get a key at https://openrouter.ai/keys  (starts with sk-or-)
2. Put in your .env:
       OPENAI_API_KEY=sk-or-...
3. Create the model like this:

   llm = ChatOpenAI(
       model="nvidia/nemotron-3-super-120b-a12b:free",
       temperature=0,
       base_url="https://openrouter.ai/api/v1",
   )

Free NVIDIA Nemotron models (the ":free" suffix is REQUIRED —
without it you'll be billed):
  nvidia/nemotron-3-super-120b-a12b:free   <- use this one
  nvidia/nemotron-3-nano-30b-a3b:free      <- fallback if rate-limited
  nvidia/nemotron-3-ultra-550b-a55b:free   <- biggest, often congested
Full list: https://openrouter.ai/collections/free-models

KNOW THE LIMITS: free models are rate-limited (~20 req/min and a
small daily cap). This lab makes ~5-10 LLM calls per run, so you
have plenty — but don't run it in a tight loop, and if you get
HTTP 429, wait a minute or switch to the nano model.

CAVEAT for Step 3: with_structured_output() needs tool/function
calling. Nemotron supports it, but if a free model ever returns
an error there, either (a) try another :free model, or (b) pass
method="json_schema" to with_structured_output.

NOTE: OpenRouter has NO embeddings endpoint. For the vector store
use InMemoryVectorStore + local HuggingFaceEmbeddings
(uv sync --group embeddings), or DeterministicFakeEmbedding —
embeddings only power the memory-retrieval bonus, not the core graph.
------------------------------------------------------------

GOTCHA: the old imports you'll find in 2023-24 tutorials
(langchain.vectorstores, langchain_community.tools.tavily_search)
are DEAD. Current homes:
  - TavilySearch:      https://docs.langchain.com/oss/python/integrations/providers/tavily
  - Chat models:       https://docs.langchain.com/oss/python/langchain/models
  - InMemoryVectorStore: langchain_core.vectorstores

NOTE: TavilySearch.invoke({"query": q}) returns a DICT — the
actual sources are under the "results" key. print() it once to see.

In [80]:
# STEP 2 — CLEAN VERSION (works in uv + no numpy)

from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.embeddings import DeterministicFakeEmbedding

# 1) LLM — OpenRouter (free NVIDIA model)
llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
)

# 2) Search tool — Tavily
search_tool = TavilySearch(max_results=5)

# 3) Embeddings + Vector Store (no numpy needed)
embeddings = DeterministicFakeEmbedding(size=1)  
vector_store = InMemoryVectorStore(embeddings)


## STEP 3 — STRUCTURED OUTPUT for the quality score
Never parse int(response.content) out of free text. Define a
Pydantic schema and use llm.with_structured_output(...) so the
model is FORCED to return valid data.

WHERE TO LOOK: https://docs.langchain.com/oss/python/langchain/structured-output
ASK YOURSELF: what does with_structured_output return — a string,
a dict, or a QualityScore object?

In [81]:
class QualityScore(BaseModel):
    """Evaluation of research quality."""
    score: int = Field(ge=1, le=10)
    reasoning: str = Field(description="One-sentence justification")

# TODO: evaluator = llm.with_structured_output(QualityScore)
evaluator = llm.with_structured_output(QualityScore)

## STEP 4 — NODES
A node is just a function: takes state, returns a PARTIAL update
(a dict with ONLY the keys it changed). LangGraph merges it in.
Do NOT mutate state in place; do NOT return the whole state.

WHERE TO LOOK: Use Graph API docs → "Define and update state".
  https://docs.langchain.com/oss/python/langgraph/use-graph-api

In [82]:
from langchain_core.messages import HumanMessage

def collect_node(state: AgentState):
    """Search the web. On retries, CHANGE the query."""
    iteration = state["iteration_count"] + 1
    query = f"{state['topic']} research iteration {iteration}"

    results = search_tool.invoke({"query": query})["results"]

    return {
        "search_query": query,
        "collected_data": results,
        "iteration_count": iteration,
        "execution_logs": [f"[collect] iteration {iteration} → {len(results)} results"]
    }


def store_memory_node(state: AgentState):
    """Memory disabled (no embeddings available)."""
    return {
        "execution_logs": ["[store_memory] skipped (no embeddings available)"]
    }


def analyze_node(state: AgentState):
    """LLM-analyze each source WITHOUT RAG (numpy not available)."""
    analyses = []

    for item in state["collected_data"]:
        content = item["content"]

        prompt = (
            f"Analyze the following research source:\n\n"
            f"{content}\n\n"
            f"Provide a concise analysis."
        )

        result = llm.invoke([HumanMessage(content=prompt)])
        analyses.append({"source": content, "analysis": result.content})

    return {
        "analyzed_data": analyses,
        "execution_logs": [f"[analyze] analyzed {len(analyses)} items"]
    }


def evaluate_node(state: AgentState):
    """Score the research with the evaluator."""
    prompt = (
        "Evaluate the quality of the following research analysis:\n\n"
        f"{state['analyzed_data']}"
    )

    result = evaluator.invoke([HumanMessage(content=prompt)])

    return {
        "quality_score": result.score,
        "execution_logs": [f"[evaluate] score = {result.score}"]
    }


def report_node(state: AgentState):
    """Generate the enterprise report."""
    sections = "\n\n".join(
        [f"Source:\n{item['source']}\n\nAnalysis:\n{item['analysis']}"
         for item in state["analyzed_data"]]
    )

    prompt = (
        f"Write a professional enterprise research report about: {state['topic']}\n\n"
        f"Use the following analyses:\n\n{sections}"
    )

    result = llm.invoke([HumanMessage(content=prompt)])

    return {
        "final_report": result.content,
        "execution_logs": ["[report] report generated"]
    }


def audit_node(state: AgentState):
    """Log completion stats."""
    log = (
        f"Audit:\n"
        f"Topic: {state['topic']}\n"
        f"Iterations: {state['iteration_count']}\n"
        f"Final score: {state['quality_score']}\n"
        f"Report length: {len(state['final_report'])} chars"
    )

    return {
        "execution_logs": [log]
    }


## STEP 5 — THE CONDITIONAL EDGE (the heart of this lab)
Write a router function: takes state, RETURNS THE NAME of the
next node as a string.

CRITICAL — loops must terminate. Two rules:
  a) every retry must change something (your query, Step 4.2),
  b) hard-cap the retries with iteration_count.
Without both, same search → same score → infinite loop → LangGraph
kills the run at recursion limit 25 with GraphRecursionError.

WHERE TO LOOK (read BOTH):
  - "Conditional branching":
    https://docs.langchain.com/oss/python/langgraph/use-graph-api#conditional-branching
  - "Create and control loops":
    https://docs.langchain.com/oss/python/langgraph/use-graph-api#create-and-control-loops

EXPERIMENT: comment out the iteration cap, force low scores, run,
and read the GraphRecursionError message. Now you understand why
the docs insist on termination conditions.

In [83]:
def quality_router(state: AgentState) -> str:
    # TODO: return "report" or "collect"

    if state["iteration_count"] >= 3:
        return "report"

    if state["quality_score"] < 7:
        return "collect"

    return "report"


## STEP 6 — WIRE THE GRAPH
1. workflow = StateGraph(AgentState)
2. add_node(...) for all six nodes
3. add_edge(START, "collect")        <- START, not set_entry_point
4. linear edges: collect → store_memory → analyze → evaluate
5. add_conditional_edges("evaluate", quality_router,
       {"collect": "collect", "report": "report"})
   (the dict maps router RETURN VALUES to NODE NAMES)
6. report → audit → END

WHERE TO LOOK: Graph API docs → "Edges".

In [84]:
# TODO: your code here
# STEP 6 — WIRE THE GRAPH

# 1) 
workflow = StateGraph(AgentState)

# 2) 
workflow.add_node("collect", collect_node)
workflow.add_node("store_memory", store_memory_node)
workflow.add_node("analyze", analyze_node)
workflow.add_node("evaluate", evaluate_node)
workflow.add_node("report", report_node)
workflow.add_node("audit", audit_node)

# 3) 
workflow.add_edge(START, "collect")

# 4) 
# 
workflow.add_edge("collect", "store_memory")
workflow.add_edge("store_memory", "analyze")
workflow.add_edge("analyze", "evaluate")

# 5) 
workflow.add_conditional_edges(
    "evaluate",
    quality_router,
    {
        "collect": "collect",
        "report": "report"
    }
)

# 6) 
workflow.add_edge("report", "audit")
workflow.add_edge("audit", END)

# 7) 
memory = InMemorySaver()
app = workflow.compile(checkpointer=memory)


## STEP 7 — COMPILE with a checkpointer, VISUALIZE, RUN
1. app = workflow.compile(checkpointer=InMemorySaver())
   A checkpointer saves state after every node → enables resume,
   time-travel debugging, and human-in-the-loop.
   WHERE TO LOOK: https://docs.langchain.com/oss/python/langgraph/persistence

2. Visualize what you built:
      print(app.get_graph().draw_mermaid())
   → paste the output into https://mermaid.live
   Does the picture match the diagram at the top of this file?

3. Run with STREAMING so you watch state evolve node by node:
      config = {"configurable": {"thread_id": "run-1"}}  # required
      for chunk in app.stream(initial_state, config,
                              stream_mode="values"):
          ...
   WHERE TO LOOK: https://docs.langchain.com/oss/python/langgraph/streaming

4. BONUS — human-in-the-loop: compile with
      interrupt_before=["report"]
   then inspect state and resume. WHERE TO LOOK:
      https://docs.langchain.com/oss/python/langgraph/interrupts

In [85]:
app = workflow.compile(checkpointer=InMemorySaver())

print(app.get_graph().draw_mermaid())

initial_state = {
    "topic": "Enterprise Agentic AI Systems",
    "search_query": "",
    "collected_data": [],
    "analyzed_data": [],
    "quality_score": 0,
    "iteration_count": 0,
    "final_report": "",
    "execution_logs": [],
}
# TODO: compile, visualize, stream, print final report + logs

# 4) 
config = {"configurable": {"thread_id": "run-1"}}

print("\n--- STREAMING OUTPUT ---\n")
for chunk in app.stream(initial_state, config, stream_mode="values"):
    print(chunk)

# 5) 
final_state = app.invoke(initial_state, config)

print("\n--- FINAL REPORT ---\n")
print(final_state["final_report"])

print("\n--- EXECUTION LOGS ---\n")
for log in final_state["execution_logs"]:
    print(log)


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	collect(collect)
	store_memory(store_memory)
	analyze(analyze)
	evaluate(evaluate)
	report(report)
	audit(audit)
	__end__([<p>__end__</p>]):::last
	__start__ --> collect;
	analyze --> evaluate;
	collect --> store_memory;
	evaluate -.-> collect;
	evaluate -.-> report;
	report --> audit;
	store_memory --> analyze;
	audit --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


--- STREAMING OUTPUT ---

{'topic': 'Enterprise Agentic AI Systems', 'search_query': '', 'collected_data': [], 'analyzed_data': [], 'quality_score': 0, 'iteration_count': 0, 'final_report': '', 'execution_logs': []}
{'topic': 'Enterprise Agentic AI Systems', 'search_query': 'Enterprise Agentic AI Systems research iteration 1', 'collected_data': [{'url': 'https://www.truefoundry.com/blog/agentic-ai-in-enterprise', 'title': 'Agentic AI in Enterprises: Scaling

## SELF-CHECK before you look at the solution
[ ] My nodes return partial dicts, never the whole mutated state
[ ] execution_logs uses a reducer, and I can explain why
[ ] My router has BOTH a quality exit AND an iteration cap
[ ] Retried searches use a different query than the first attempt
[ ] I saw the Mermaid diagram and it matches the intended flow
[ ] I know what GraphRecursionError is and how to trigger it
[ ] The quality score comes from with_structured_output, not int()

Stuck? Debugging order that works:
  1. print() the raw return of search_tool.invoke — check its shape
  2. run app.stream(..., stream_mode="updates") — shows exactly
     which node produced which state update
  3. compare your edge wiring against the diagram at the top
  4. only THEN open day1_lab_solution.py